# Batch Evaluation & Semantic Re-ranking
This notebook runs the end-to-end evaluation of the Visual Product Search Engine on Kaggle.
It computes Recall@K, NDCG@K, and mAP@K for the 3 ablation conditions:
- **A**: Vision-only CLIP ($alpha=1.0$)
- **B**: Frozen CLIP + Frozen BLIP-2
- **C**: Fine-tuned CLIP + Frozen BLIP-2


In [1]:
!pip install -q omegaconf timm fairscale iopath decord webdataset pycocotools pycocoevalcap
!pip install -q --no-dependencies salesforce-lavis
!pip install -q transformers==4.38.2 open_clip_torch pinecone pandas Pillow tqdm scikit-learn accelerate

In [2]:
import os
import torch
import open_clip
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pinecone import Pinecone
from lavis.models import load_model_and_preprocess
import numpy as np
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__

Device: cuda


In [3]:
# ==============================================================================
# CONFIGURATIONS
# ==============================================================================
PINECONE_API_KEY  = "pcsk_REDACTED"  # Replace with your key
INDEX_NAME        = "vr-clothing-gallery"

DATASET_ROOT      = "/kaggle/input/datasets/vinay1706/deepfashion-cropped" # Verify path
CLIP_CHECKPOINT   = f"{DATASET_ROOT}/clip_best.pt"
QUERY_CSV         = f"{DATASET_ROOT}/query.csv"
GALLERY_CSV       = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV      = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT        = DATASET_ROOT

# Top-K settings
TOP_K_RETRIEVAL = 15
K_VALUES = [5, 10, 15]

# Ablation Configs to Evaluate: (Config Name, Namespace, Use Finetuned CLIP, Use BLIP-2 Reranking)
CONFIGS = [
    ("Config A (Vision-Only)", "frozen-alpha-1.0", False, False),
    ("Config B (Frozen CLIP, alpha=0.7)", "frozen-alpha-0.7", False, True),
    ("Config B (Frozen CLIP, alpha=0.5)", "frozen-alpha-0.5", False, True),
    ("Config C (Finetuned CLIP, alpha=0.7)", "finetuned-alpha-0.7", True, True),
    ("Config C (Finetuned CLIP, alpha=0.5)", "finetuned-alpha-0.5", True, True)
]


In [4]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
query_df = pd.read_csv(QUERY_CSV)
gallery_df = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

# Create a mapping from image_name to caption for the gallery
caption_map = dict(zip(captions_df['image_name'], captions_df['blip2_caption']))

print(f"Loaded {len(query_df)} query images.")
print(f"Loaded {len(gallery_df)} gallery images.")


Loaded 14218 query images.
Loaded 12612 gallery images.


In [5]:
# ==============================================================================
# LOAD MODELS
# ==============================================================================
print("Loading LAVIS BLIP-2 ITM Model...")
blip_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="blip2_image_text_matching", 
    model_type="pretrain", 
    is_eval=True, 
    device=device
)

# Fix LAVIS missing padding in batched inference
class PatchedTokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, *args, **kwargs):
        kwargs["padding"] = True
        return self.tokenizer(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

blip_model.tokenizer = PatchedTokenizer(blip_model.tokenizer)

# CLIP loading helper
def load_clip(use_finetuned):
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    if use_finetuned:
        ckpt = torch.load(CLIP_CHECKPOINT, map_location=device)
        # BUG FIX: The checkpoint uses the key 'model_state', not 'model_state_dict'
        state_dict = ckpt.get("model_state", ckpt)
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
    return model.to(device).eval(), preprocess

print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


Loading LAVIS BLIP-2 ITM Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Connecting to Pinecone...


In [6]:
# ==============================================================================
# BLIP-2 ITM SCORING (LAVIS Batch Method)
# ==============================================================================
def compute_itm_scores_batched(raw_image, candidate_captions):
    """
    Computes true ITM probabilities for a batch of candidate captions against a single image.
    Returns a list of probabilities (0.0 to 1.0).
    """
    # Preprocess image once
    img = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
    
    # Duplicate image tensor to match the batch size of captions
    img_batch = img.repeat(len(candidate_captions), 1, 1, 1)
    
    # Preprocess all candidate text captions
    txt_batch = [txt_processors["eval"](c) for c in candidate_captions]
    
    # Pass through the ITM model in a single batch
    with torch.no_grad():
        itm_output = blip_model({"image": img_batch, "text_input": txt_batch}, match_head="itm")
        
        # The model outputs logits for [Not Match, Match]. 
        # We take the softmax and extract the probability of Match (index 1).
        itm_scores = torch.nn.functional.softmax(itm_output, dim=1)[:, 1].cpu().tolist()
        
    return itm_scores


In [7]:
# ==============================================================================
# METRICS CALCULATION
# ==============================================================================
def calculate_metrics(ranked_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    metrics = {}
    for k in k_values:
        top_k_items = ranked_item_ids[:k]
        
        # Recall@K: 1 if ground truth is in top-K, else 0
        recall = 1 if ground_truth_id in top_k_items else 0
        metrics[f'Recall@{k}'] = recall
        
        # NDCG@K
        dcg = 0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                dcg += 1 / math.log2(i + 2)
                break
                
        # Ideal DCG is 1 because we only consider the first matching item
        idcg = 1 
        metrics[f'NDCG@{k}'] = dcg / idcg
        
        # mAP@K (Average Precision)
        # Since there is only 1 relevant item, AP is just 1/(rank) if found, else 0
        ap = 0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                ap = 1 / (i + 1)
                break
        metrics[f'mAP@{k}'] = ap
        
    return metrics


In [8]:
# ==============================================================================
# MAIN EVALUATION LOOP
# ==============================================================================

# Subset for testing (set to None for full evaluation)
NUM_QUERIES_TO_EVALUATE = 50 

if NUM_QUERIES_TO_EVALUATE:
    eval_df = query_df.head(NUM_QUERIES_TO_EVALUATE)
else:
    eval_df = query_df

results_summary = []

for config_name, namespace, use_finetuned, use_reranking in CONFIGS:
    print(f"\n{'='*60}\nEvaluating {config_name}\n{'='*60}")
    
    # Load appropriate CLIP model
    clip_model, clip_preprocess = load_clip(use_finetuned)
    
    config_metrics = {f'Recall@{k}': [] for k in K_VALUES}
    config_metrics.update({f'NDCG@{k}': [] for k in K_VALUES})
    config_metrics.update({f'mAP@{k}': [] for k in K_VALUES})
    
    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        query_image_path = os.path.join(IMAGE_ROOT, row['image_name'])
        gt_item_id = row['item_id']
        
        # 1. Load and process query image
        try:
            raw_image = Image.open(query_image_path).convert('RGB')
            # Assuming images are already cropped in this dataset.
            processed_image = clip_preprocess(raw_image).unsqueeze(0).to(device)
        except Exception as e:
            continue
            
        # 2. Get CLIP Visual Embedding
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                query_embedding = clip_model.encode_image(processed_image)
                query_embedding = torch.nn.functional.normalize(query_embedding, dim=-1).cpu().numpy().tolist()[0]
                
        # 3. Retrieve Top-K candidates from Pinecone
        retrieval_response = index.query(
            vector=query_embedding,
            top_k=TOP_K_RETRIEVAL,
            namespace=namespace,
            include_metadata=True
        )
        
        candidates = retrieval_response['matches']
        
        # 4. Semantic Re-ranking
        if use_reranking:
            cand_images = [cand['metadata']['image_name'] for cand in candidates]
            cand_captions = [caption_map.get(img, "") for img in cand_images]
            
            # Get ITM Scores in one batch
            itm_scores = compute_itm_scores_batched(raw_image, cand_captions)
            
            scored_candidates = list(zip(candidates, itm_scores))
                
            # Re-sort candidates by ITM score (descending probability)
            scored_candidates.sort(key=lambda x: x[1], reverse=True)
            candidates = [item[0] for item in scored_candidates]
            
        # 5. Extract Ranked Item IDs
        ranked_item_ids = [cand['metadata']['item_id'] for cand in candidates]
        
        # 6. Calculate Metrics
        q_metrics = calculate_metrics(ranked_item_ids, gt_item_id, K_VALUES)
        
        for k, v in q_metrics.items():
            config_metrics[k].append(v)
            
    # Aggregate and print results
    print(f"\nResults for {config_name}:")
    summary_row = {"Config": config_name}
    for metric, values in config_metrics.items():
        mean_val = np.mean(values)
        summary_row[metric] = mean_val
        print(f"{metric}: {mean_val:.4f}")
        
    results_summary.append(summary_row)
    
    # Clean up CLIP model memory before next config
    del clip_model
    torch.cuda.empty_cache()

# Display Final Summary Table
results_df = pd.DataFrame(results_summary)
display(results_df)



Evaluating Config A (Vision-Only)


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  0%|          | 0/50 [00:00<?, ?it/s]


Results for Config A (Vision-Only):
Recall@5: 0.5400
Recall@10: 0.5800
Recall@15: 0.6000
NDCG@5: 0.4733
NDCG@10: 0.4858
NDCG@15: 0.4910
mAP@5: 0.4513
mAP@10: 0.4562
mAP@15: 0.4577

Evaluating Config B (Frozen CLIP, alpha=0.7)


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lavis/models/blip2_models/blip2.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(dtype=dtype)



Results for Config C (Finetuned CLIP, alpha=0.5):
Recall@5: 0.6600
Recall@10: 0.7400
Recall@15: 0.7400
NDCG@5: 0.5641
NDCG@10: 0.5904
NDCG@15: 0.5904
mAP@5: 0.5323
mAP@10: 0.5434
mAP@15: 0.5434


,Config,Recall@5,Recall@10,Recall@15,NDCG@5,NDCG@10,NDCG@15,mAP@5,mAP@10,mAP@15
0,Config A (Vision-Only),0.54,0.58,0.60,0.473330,0.485778,0.491031,0.451333,0.456190,0.457729
1,"Config B (Frozen CLIP, alpha=0.7)",0.58,0.64,0.66,0.491707,0.510175,0.515175,0.461667,0.468746,0.470079
2,"Config B (Frozen CLIP, alpha=0.5)",0.60,0.66,0.68,0.467145,0.486717,0.491970,0.422333,0.430524,0.432062
3,"Config C (Finetuned CLIP, alpha=0.7)",0.72,0.78,0.82,0.610830,0.630641,0.641799,0.574000,0.582413,0.586049
4,"Config C (Finetuned CLIP, alpha=0.5)",0.66,0.74,0.74,0.564052,0.590361,0.590361,0.532333,0.543405,0.543405
